# Load EURUSD Data From Database

Scratch notebook for pulling raw daily candles from `fx_candles.db` and eyeballing
recent data before building indicators on top of it.

- Fetches every stored daily candle for `INSTRUMENT` in `YEAR`.
- Prints full detail for the most recent `LAST_N_DAYS` trading days.
- Re-run all cells to refresh against the latest data in `fx_candles.db`.

In [1]:
from __future__ import annotations

import sqlite3
from typing import Final

import pandas as pd

In [ ]:
DB_PATH: Final[str] = "../../fx_candles.db"
TABLE: Final[str] = "candles_D"
INSTRUMENT: Final[str] = "EUR_USD"
YEAR: Final[int] = 2026
LAST_N_DAYS: Final[int] = 30

In [3]:
query = (
    f"SELECT * FROM {TABLE} "
    "WHERE instrument = ? AND time >= ? AND time < ? AND complete = 1 "
    "ORDER BY time ASC"
)
with sqlite3.connect(DB_PATH) as conn:
    candles = pd.read_sql_query(
        query, conn, params=(INSTRUMENT, f"{YEAR}-01-01", f"{YEAR + 1}-01-01")
    )

# print(candles.head())
print(f"Fetched {len(candles)} {INSTRUMENT} candles for {YEAR}.")

DatabaseError: Execution failed on sql 'SELECT * FROM candles_D WHERE instrument = ? AND time >= ? AND time < ? AND complete = 1 ORDER BY time ASC': no such table: candles_D

In [ ]:
candles["date"] = pd.to_datetime(candles["time"]).dt.date

# Get all unique dates in the candles DF -> sort them chronologically -> take the last N dates of them
last_30_dates = sorted(candles["date"].unique())[-LAST_N_DAYS:]
last_30_days = candles[candles["date"].isin(last_30_dates)]
# print(last_30_days)

print(f"Last {len(last_30_dates)} trading days: {last_30_dates[0]} → {last_30_dates[-1]}")

In [ ]:
with pd.option_context("display.max_rows", None, "display.max_columns", None,
                        "display.width", None):
    print(last_30_days.to_string(index=False))